In [ ]:

import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from bed_reader import open_bed

CIGWAS_PATH = Path("<PATH_TO_CI_GWAS_FULL_RESULTS_TSV>")
STEP2_ROOT  = Path("<PATH_TO_REGENIE_STEP2_ROOT>")

PLINK_ROOT = Path("<PATH_TO_PLINK_ROOT>")
PHENO_ROOT = Path("<PATH_TO_REGENIE_PHENO_ROOT>")

OUTDIR = Path("<PATH_TO_DRUG_VARIANT_TABLES_DIR>")
OUTDIR.mkdir(parents=True, exist_ok=True)

OUT_TSV = OUTDIR / "<PATH_TO_DRUG_VARIANTS_REGENIE_LOOKUP_WITH_CARRIERS_TSV>"

MAF_THRESHOLD = 5e-4

MAIN_SETUPS = [
    "sbp_pre_post_1to60_no_cvd",
    "dbp_pre_post_1to60_no_cvd",
    "sbp_pre_post_1to60_age5_with_statins_no_cvd",
    "dbp_pre_post_1to60_age5_with_statins_no_cvd",
    "sbp_pre_post_1to60_on_drugs_no_cvd",
    "dbp_pre_post_1to60_on_drugs_no_cvd",
]

DRUGS = [
    "ACE_inhibitor",
    "angiotensin_receptor_blocker",
    "beta_blocker",
    "calcium_channel_blocker",
    "diuretic",
    "statin",
]

_age_re = re.compile(r"_([1-5])_ADJ$", re.IGNORECASE)

def infer_trait_from_setup(setup):
    s = str(setup).lower()
    if "sbp" in s:
        return "SBP"
    if "dbp" in s:
        return "DBP"
    raise ValueError(f"Cannot infer trait from setup={setup!r}")

def infer_age_bin(phenotype):
    m = _age_re.search(str(phenotype))
    return int(m.group(1)) if m else None

def is_drug_phenotype(phenotype):
    p = str(phenotype).lower()
    return any(d.lower() in p for d in DRUGS)

def load_ci_drug_hits():
    df = pd.read_csv(CIGWAS_PATH, sep="\t")
    df["setup"] = df["setup"].astype(str)
    df["phenotype"] = df["phenotype"].astype(str)
    df["rsID"] = df["rsID"].astype(str)

    df = df[df["setup"].isin(MAIN_SETUPS)].copy()
    df = df[df["p_fdr"] < 0.05].copy()
    df = df[df["phenotype"].map(is_drug_phenotype)].copy()

    df["chr"] = pd.to_numeric(df["chr"], errors="coerce")
    df = df[df["chr"].notna()].copy()
    df["chr"] = df["chr"].astype(int)

    df["bp"] = pd.to_numeric(df["bp"], errors="coerce").astype("Int64")
    return df

def pooled_core_runs(trait):
    # baseline + drug response (delta only; with/without baseline adjustment)
    return [
        f"pooled__{trait}_pre__base",
        f"pooled__{trait}_delta__base+classes",
        f"pooled__{trait}_delta__base+classes+pre",
    ]

def age5_core_runs(trait, k):
    return [
        f"age5g{k}__{trait}_pre__base",
        f"age5g{k}__{trait}_delta__base+classes",
        f"age5g{k}__{trait}_delta__base+classes+pre",
    ]

def pooled_intx_delta_only_runs(trait):
    out = []
    for drug in DRUGS:
        out.append(f"intx__{trait}_delta__INT_{drug}__base+classes")
        out.append(f"intx__{trait}_delta__INT_{drug}__base+classes+pre")
    return out

def runs_for_ci_row(setup, phenotype):
    # age-split CI rows -> age5 core runs + pooled interaction runs
    trait = infer_trait_from_setup(setup)
    k = infer_age_bin(phenotype)
    if k is not None:
        return age5_core_runs(trait, k) + pooled_intx_delta_only_runs(trait)
    return pooled_core_runs(trait) + pooled_intx_delta_only_runs(trait)

def trait_name_from_run(run_id):
    return run_id.split("__")[1]

def wanted_test_for_run(run_id):
    if run_id.startswith("intx__"):
        drug = run_id.split("__")[2].replace("INT_", "")
        return f"ADD-INT_SNPx{drug}"
    return "ADD"

def step2_file(run_id, chrom, trait_name):
    return STEP2_ROOT / run_id / f"chr{chrom}" / f"{run_id}_chr{chrom}_step2_{trait_name}.regenie"

def scan_step2(path, ids, want_test):
    out = {}

    with path.open() as f:
        header = f.readline().strip().split()

        def col_idx(cands):
            u = {c.upper(): i for i, c in enumerate(header)}
            for c in cands:
                if c.upper() in u:
                    return u[c.upper()]
            raise KeyError(f"None of {cands} found in header of {path}. Have: {header}")

        i_id    = col_idx(["ID"])
        i_af    = col_idx(["A1FREQ"])
        i_test  = col_idx(["TEST"])
        i_beta  = col_idx(["BETA"])
        i_log10 = col_idx(["LOG10P"])

        for ln in f:
            c = ln.split()
            vid = c[i_id]
            if vid not in ids:
                continue
            if c[i_test] != want_test:
                continue

            b, lp, af = c[i_beta], c[i_log10], c[i_af]
            if b == "NA" or lp == "NA" or af == "NA":
                continue

            af = float(af)
            maf = af if af <= 0.5 else (1.0 - af)

            out[vid] = {"beta": float(b), "log10p": float(lp), "maf": maf}
            if len(out) == len(ids):
                break

    return out

def describe_run(run_id):
    parts = run_id.split("__")
    prefix = parts[0]
    trait_kind = parts[1]
    trait, kind = trait_kind.split("_", 1)

    scope = prefix
    interaction = pd.NA
    cov_token = parts[-1]

    if prefix == "intx":
        scope = "pooled_intx"
        interaction = parts[2].replace("INT_", "")
        cov_token = parts[3]

    tokens = cov_token.split("+")
    cov = ["basic covariates and confounders"]
    if "classes" in tokens:
        cov.append("other drug classes" if prefix == "intx" else "all drug classes")
    if "pre" in tokens:
        cov.append(f"{trait}-pre")

    return {
        "gwas_scope": scope,
        "gwas_phenotype": f"{kind} {trait}",
        "interaction": interaction,
        "covariates": " + ".join(cov),
    }

_BIM = {}
_FAM_IID2IX = {}
_PHEN_IIDS = {}

def load_bim(chrom):
    if chrom not in _BIM:
        p = PLINK_ROOT / f"c{chrom}.bim"
        _BIM[chrom] = pd.read_csv(
            p, sep=r"\s+", header=None,
            names=["CHR","ID","CM","BP","A1","A2"],
            dtype={"CHR": int, "ID": str, "CM": float, "BP": int, "A1": str, "A2": str},
        )
    return _BIM[chrom]

def fam_iid2ix(chrom):
    if chrom not in _FAM_IID2IX:
        p = PLINK_ROOT / f"c{chrom}.fam"
        fam = pd.read_csv(p, sep=r"\s+", header=None, usecols=[1], names=["IID"], dtype=str)
        _FAM_IID2IX[chrom] = {iid: i for i, iid in enumerate(fam["IID"].astype(str).values)}
    return _FAM_IID2IX[chrom]

def variant_ix(chrom, rsid, bp):
    bim = load_bim(chrom)
    return int(bim.index[bim["ID"].eq(str(rsid))][0])

def maf_all_samples(chrom, vix):
    bed_path = PLINK_ROOT / f"c{chrom}.bed"
    with open_bed(bed_path) as bed:
        g = bed.read(index=np.s_[:, vix]).reshape(-1)

    nonmiss = ~np.isnan(g)
    n = int(nonmiss.sum())
    if n == 0:
        return pd.NA

    af = float(np.nansum(g)) / (2.0 * n)
    return af if af <= 0.5 else (1.0 - af)

def tested_iids(run_id):
    if run_id not in _PHEN_IIDS:
        phen_path = PHENO_ROOT / f"{run_id}.phen"
        df = pd.read_csv(phen_path, sep=r"\s+", dtype=str, keep_default_na=False)
        ph_col = df.columns[2]
        vals = pd.to_numeric(df[ph_col].replace("NA", np.nan), errors="coerce")
        _PHEN_IIDS[run_id] = df.loc[vals.notna(), "IID"].astype(str).values
    return _PHEN_IIDS[run_id]

def carrier_stats(chrom, vix, sample_ix):
    if sample_ix.size == 0:
        return {"n_geno_nonmissing": 0, "n_carriers": 0, "n_het": 0, "n_hom_alt": 0}

    bed_path = PLINK_ROOT / f"c{chrom}.bed"
    with open_bed(bed_path) as bed:
        g = bed.read(index=np.s_[sample_ix, vix]).reshape(-1)

    nonmiss = ~np.isnan(g)
    g2 = g[nonmiss]
    return {
        "n_geno_nonmissing": int(nonmiss.sum()),
        "n_carriers": int(np.sum(g2 > 0)),
        "n_het": int(np.sum(g2 == 1)),
        "n_hom_alt": int(np.sum(g2 == 2)),
    }

cigwas = load_ci_drug_hits()

rows = cigwas[["setup","phenotype","rsID","chr","bp","p_fdr","p_value"]].copy()
rows["trait"] = rows["setup"].map(infer_trait_from_setup)
rows["age_bin"] = rows["phenotype"].map(infer_age_bin)
rows["runs"] = [runs_for_ci_row(s, p) for s, p in zip(rows["setup"], rows["phenotype"])]

need = defaultdict(set)
for r in rows.itertuples(index=False):
    for run in r.runs:
        need[(run, int(r.chr))].add(str(r.rsID))

found = {}

for (run, chr_), ids in need.items():
    trait_name = trait_name_from_run(run)
    want_test = wanted_test_for_run(run)
    fpath = step2_file(run, chr_, trait_name)
    hit = scan_step2(fpath, ids, want_test)
    for vid, d in hit.items():
        found[(run, chr_, vid)] = d

records = []
for r in rows.itertuples(index=False):
    for run_id in r.runs:
        d = found.get((run_id, int(r.chr), str(r.rsID)))
        meta = describe_run(run_id)

        records.append({
                "setup": r.setup,
                "phenotype": r.phenotype,  # drug-class phenotype from CI-GWAS
                "trait": r.trait,          # SBP/DBP inferred from setup
                "age_bin": r.age_bin,

                "rsID": str(r.rsID),
                "chr": int(r.chr),
                "bp": (int(r.bp) if pd.notna(r.bp) else pd.NA),
                "ci_p_value": float(r.p_value) if pd.notna(r.p_value) else pd.NA,
                "ci_p_fdr": float(r.p_fdr) if pd.notna(r.p_fdr) else pd.NA,

                "gwas_scope": meta["gwas_scope"],
                "gwas_phenotype": meta["gwas_phenotype"],
                "interaction": meta["interaction"],
                "covariates": meta["covariates"],

                "beta": (d["beta"] if d else pd.NA),
                "log10p": (d["log10p"] if d else pd.NA),
                "MAF": (d["maf"] if d else pd.NA),
                "run_id": run_id,
        })

res = pd.DataFrame.from_records(records)
res["bp"] = pd.to_numeric(res["bp"], errors="coerce").astype("Int64")

res["p"] = pd.NA
m = res["log10p"].notna()
res.loc[m, "p"] = 10 ** (-pd.to_numeric(res.loc[m, "log10p"], errors="coerce"))

In [ ]:
maf_num = pd.to_numeric(res["MAF"], errors="coerce")

# carrier annotations ONLY for rare variants
rare_mask = maf_num.notna() & (maf_num < MAF_THRESHOLD)
targets = res.loc[rare_mask, ["run_id", "chr", "bp", "rsID"]].drop_duplicates()

stats_rows = []
for t in targets.itertuples(index=False):
    run_id = str(t.run_id)
    chrom = int(t.chr)
    bp = (int(t.bp) if pd.notna(t.bp) else None)
    rsid = str(t.rsID)

    vix = variant_ix(chrom, rsid, bp)
    if vix is None:
        stats_rows.append({
            "run_id": run_id, "chr": chrom, "bp": t.bp, "rsID": rsid,
            "n_tested": pd.NA, "n_geno_nonmissing": pd.NA, "n_carriers": pd.NA,
            "n_het": pd.NA, "n_hom_alt": pd.NA
        })
        continue

    iids = tested_iids(run_id)
    iid2ix = fam_iid2ix(chrom)
    sample_ix = np.array([iid2ix[i] for i in iids if i in iid2ix], dtype=int)

    st = carrier_stats(chrom, vix, sample_ix)
    stats_rows.append({
        "run_id": run_id, "chr": chrom, "bp": t.bp, "rsID": rsid,
        "n_tested": int(sample_ix.size),
        **st
    })

stats_df = pd.DataFrame(stats_rows)
res = res.merge(stats_df, on=["run_id", "chr", "bp", "rsID"], how="left")

res.to_csv(OUT_TSV, sep="\t", index=False, na_rep="NA")

print("Saved:", OUT_TSV)
print("CI drug hits:", cigwas[["setup", "phenotype", "rsID"]].drop_duplicates().shape[0])
print("Output rows:", res.shape[0])
print("Rare rows (MAF < threshold):", int(rare_mask.sum()))
print("Rare targets (unique run/variant):", targets.shape[0])